In [16]:
# 1. 必要なライブラリのインストール（Gradioだけなので一瞬で終わります）
!pip install -q gradio

import gradio as gr

# 2. 条件（性別×系統×気温）に応じたイラストURLのデータベース
# ※ここではサンプルとして公開されているイラストURLを指定しています。
# ※自分で好きなイラスト（ネット上のURL）に差し替えることも簡単にできます！
IMAGE_DATABASE = {
    "女性": {
        "カジュアル": {
            "冬": "https://images.unsplash.com/photo-1548624313-0396c75e4b1a?w=500&q=80", # ニット帽と冬服の女性
            "春秋": "https://images.unsplash.com/photo-1515886657613-9f3515b0c78f?w=500&q=80", # 明るいカジュアル服の女性
            "夏": "https://images.unsplash.com/photo-1503342217505-b0a15ec3261c?w=500&q=80" # Tシャツ姿の女性
        },
        "きれいめ・オフィスカジュアル": {
            "冬": "https://images.unsplash.com/photo-1485968579580-b6d095142e6e?w=500&q=80",
            "春秋": "https://images.unsplash.com/photo-1494790108377-be9c29b29330?w=500&q=80",
            "夏": "https://images.unsplash.com/photo-1524504388940-b1c1722653e1?w=500&q=80"
        }
    },
    "男性": {
        "カジュアル": {
            "冬": "https://images.unsplash.com/photo-1507679799987-c73779587ccf?w=500&q=80", # スーツ・コートの男性
            "春秋": "https://images.unsplash.com/photo-1519085360753-af0119f7cbe7?w=500&q=80", # シャツ姿の男性
            "夏": "https://images.unsplash.com/photo-1492562080023-ab3db95bfbce?w=500&q=80" # さわやかな夏服の男性
        },
        "きれいめ・オフィスカジュアル": {
            "冬": "https://images.unsplash.com/photo-1519345182560-3f2917c472ef?w=500&q=80",
            "春秋": "https://images.unsplash.com/photo-1489980508314-941910ded1f4?w=500&q=80",
            "夏": "https://images.unsplash.com/photo-1500648767791-00dcc994a43e?w=500&q=80"
        }
    }
}

# 3. メイン処理関数
def coordinate_propose(gender, style, thermal_type, temperature, humidity, weather):
    # 「指定なし」「ストリート」「モード」「フェミニン」が選ばれた場合の裏側の割り当て（エラー防止）
    select_gender = "男性" if gender == "男性" else "女性"
    select_style = "きれいめ・オフィスカジュアル" if style in ["きれいめ・オフィスカジュアル", "モード", "フェミニン/トラッド"] else "カジュアル"

    # 体感温度の計算
    perceived_temp = temperature
    if thermal_type == "暑がり": perceived_temp += 2
    elif thermal_type == "寒がり": perceived_temp -= 2
    if humidity > 70: perceived_temp += 1

    # 季節区分の判定
    if perceived_temp < 13:
        season = "冬"
        outer, tops, bottoms = "厚手のコートやダウン", "タートルネックニット", "厚手のロングパンツ"
    elif perceived_temp < 23:
        season = "春秋"
        outer, tops, bottoms = "ジャケットやカーディガン", "長袖シャツ・カットソー", "デニムやスラックス"
    else:
        season = "夏"
        outer, tops, bottoms = "不要（日傘など）", "半袖Tシャツやリネンシャツ", "通気性の良いパンツ"

    # テキストの作成
    result_text = f"""
### 💡 今日のコーディネート提案 ({gender} - {style}系統)
* **おすすめの季節感:** {season}の装い
* **アウター:** {outer}
* **トップス:** {tops}
* **ボトムス:** {bottoms}

---
天気は **{weather}** 、気温は **{temperature}℃** です。素敵な一日をお過ごしください！
"""

    # データベースから該当するイラストのURLを取得
    try:
        image_url = IMAGE_DATABASE[select_gender][select_style][season]
    except:
        # 万が一見つからなかった場合の予備画像
        image_url = "https://images.unsplash.com/photo-1523381210434-271e8be1f52b?w=500&q=80"

    return result_text, image_url

# 4. UIの構築
with gr.Blocks(title="お天気コーディネーター") as demo:
    gr.Markdown("# 🌤️ 服装コーディネート提案アプリ（確実作動版）")
    gr.Markdown("条件を入力してボタンを押すと、お天気に合わせたコーディネートの解説と**イメージイラスト（写真）**が表示されます。")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 👤 1. ユーザー登録情報")
            gender = gr.Radio(["女性", "男性", "指定なし"], label="性別", value="女性")
            style = gr.Dropdown(["カジュアル", "きれいめ・オフィスカジュアル", "ストリート", "モード", "フェミニン/トラッド"], label="服の系統", value="カジュアル")
            thermal_type = gr.Radio(["暑がり", "普通", "寒がり"], label="体質", value="普通")

            gr.Markdown("### 🌦️ 2. 今日の天気・環境")
            weather = gr.Dropdown(["晴れ", "曇り", "雨", "雪"], label="天気", value="晴れ")
            temperature = gr.Slider(minimum=-10, maximum=40, value=20, step=1, label="気温 (℃)")
            humidity = gr.Slider(minimum=0, maximum=100, value=50, step=5, label="湿度 (%)")

            btn = gr.Button("コーディネートをチェック", variant="primary")

        with gr.Column(scale=1):
            gr.Markdown("### 👗 提案結果")
            output_text = gr.Markdown("ここに提案テキストが表示されます。")
            # 引数に画像URLを渡すだけで、Gradioが自動でネットから読み込んで表示してくれます
            output_image = gr.Image(label="コーディネートイメージ")

        btn.click(
            fn=coordinate_propose,
            inputs=[gender, style, thermal_type, temperature, humidity, weather],
            outputs=[output_text, output_image]
        )

    demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c4cdd68aaaefb07e93.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
